# Working with Text Data

Natural Langugage processing - using machine learning and large datasets to
give computers the ability not to understand language, which is a more lofty goal, but
to ingest a piece of language as input and return something useful, like predicting the
following :

<pre>
“What’s the topic of this text?” (text classification)
“Does this text contain abuse?” (content filtering)
“Does this text sound positive or negative?” (sentiment analysis)
“What should be the next word in this incomplete sentence?” (language modeling)
“How would you say this in German?” (translation)
“How would you summarize this article in the paragraph ? (summarization)
</pre>

If CV is pattern recognition applied to pixels, NLP is pattern recognition applied to words, sentences and paragraphs

### Preparing text data

DL architecture cannot process raw text. 

Vectorization is a process of transforming raw text into numeric tensors.

Granular steps :

Standardization - process of converting to lowercase or removing punctuation.

Tokenization - Splitting text into units, such as words, characters or groups.

Indexing - converting token to a numerical index.

<img src="./images/dl_rnn_8.png" />

### Text Standardization

Two different sentences carrying the same meaning:

“sunset came. i was staring at the Mexico sky. Isnt nature splendid??”

“Sunset came; I stared at the México sky. Isn’t nature splendid?”

It's a form of feature engineering.

1) Convert to lowercase and remove punctuation

“sunset came i was staring at the mexico sky isnt nature splendid”

“sunset came i stared at the méxico sky isnt nature splendid”

2) Convert special characters to standard form / Remove numbers / Remove extra whitespace 

3) Stemming - Converting variations of a term into a single shared representation

"caught" / "been catching" -> "catch"

"was staring" / "stared" -> "stare"

4) Lemmatization - Reduces words to dictionary form

“better” → “good”

5) Stopword removal - Remove common words (“the”, “is”, “and”) when they don’t add meaning.

Intermediate representation:

“sunset came i stare at the mexico sky isnt nature splendid”

### Tokenization

#1 - Word-level / sub-word level : staring => star+ing, called => call+ed

#2 - N-gram : Where tokens are groups of N consecutive words

#3 - Character level : Each char is its own token

Two kinds of text-processing models:

Sequence models (those that care about word order) - Word level tokenization

Bag-of-words models (those that doesn't car about word order) - N-gram tokenization

##### N-gram

"the cat sat on the mat"

2-grams :

{"the", "the cat", "cat", "cat sat", "sat", "sat on", "on", "on the", "the mat", "mat"}

3-grams :

{"the", "the cat", "cat", "cat sat", "the cat sat", "sat", "sat on", "on", "cat sat on", "on the", "sat on the", "the mat", "mat", "on the mat"}

The term “bag”
here refers to the fact that you’re dealing with a set of tokens rather than a list or
sequence.

This family of tokenization methods is
called bag-of-words (or bag-of-N-grams).


Because bag-of-words isn’t an order-preserving tokenization method (the tokens gen-
erated are understood as a set, not a sequence, and the general structure of the sen-
tences is lost), it tends to be used in shallow language-processing models rather than
in deep learning models.

### Vocabulary indexing (list of all unique words)

<img src="./images/dl_rnn_10.png" />

<img src="./images/dl_rnn_9.png" />

Next logical step is to restrict the vocabulary to top 20,000 or 30,000 words.

Indexing rare terms would result in an excessively large feature space,
where most features would have almost no information content

"out of vocabulary” index (OOV) - vocabulary.get(token, 1)

Mask token (0) - reserved and used for padding

<pre>
If you want to make a batch of data with
the sequences [5, 7, 124, 4, 89] and [8, 34, 21], it would have to look like this:
[[5, 7, 124, 4, 89]
[8, 34, 21, 0, 0]]
</pre>

In [7]:
import string

class Vectorizer:
    def standardize(self, text):
        text = text.lower()
        return "".join(char for char in text if char not in string.punctuation)

    def tokenize(self, text):
        text = self.standardize(text)
        return text.split()

    def make_vocabulary(self, dataset):
        self.vocabulary = {"": 0, "[UNK]": 1}  # PAD = 0, UNK = 1
        for text in dataset:
            text = self.standardize(text)
            tokens = self.tokenize(text)
            print(tokens)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)

        # Reverse mapping: index -> token
        self.inverse_vocabulary = {v: k for k, v in self.vocabulary.items()}

    def encode(self, text):
        text = self.standardize(text)
        tokens = self.tokenize(text)
        return [self.vocabulary.get(token, 1) for token in tokens]  # 1 = [UNK]

    def decode(self, int_sequence):
        return " ".join(self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

In [61]:
dataset = [
    "Hello World!",
    "Machine Learning is fun",
    "Hello AI"
]

vec = Vectorizer()
vec.make_vocabulary(dataset)

print("Vocabulary:", vec.vocabulary)

encoded = vec.encode("Hello AI world")
print("Encoded:", encoded)

decoded = vec.decode(encoded)
print("Decoded:", decoded)

['hello', 'world']
['machine', 'learning', 'is', 'fun']
['hello', 'ai']
Vocabulary: {'': 0, '[UNK]': 1, 'hello': 2, 'world': 3, 'machine': 4, 'learning': 5, 'is': 6, 'fun': 7, 'ai': 8}
Encoded: [2, 8, 3]
Decoded: hello ai world


##### Using Keras util

In [9]:
import re
import string
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Custom standardization function
def custom_standardization_fn(string_tensor):
    # Convert to lowercase
    lowercase_string = tf.strings.lower(string_tensor)
    # Remove punctuation
    return tf.strings.regex_replace(
        lowercase_string,
        f"[{re.escape(string.punctuation)}]",
        ""
    )

# Custom split function (split on whitespace)
def custom_split_fn(string_tensor):
    return tf.strings.split(string_tensor)

# Create TextVectorization layer
text_vectorization = TextVectorization(
    output_mode="int",
    standardize=custom_standardization_fn,
    split=custom_split_fn
)

In [10]:
dataset = [
"I write, erase, rewrite",
"Erase again, and then",
"A poppy blooms.",
]
text_vectorization.adapt(dataset)

In [11]:
text_vectorization.get_vocabulary()

['',
 '[UNK]',
 np.str_('erase'),
 np.str_('write'),
 np.str_('then'),
 np.str_('rewrite'),
 np.str_('poppy'),
 np.str_('i'),
 np.str_('blooms'),
 np.str_('and'),
 np.str_('again'),
 np.str_('a')]

In [14]:
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = text_vectorization(test_sentence)

print(encoded_sentence)

tf.Tensor([ 7  3  5  9  1  5 10], shape=(7,), dtype=int64)


In [15]:
vocabulary = text_vectorization.get_vocabulary()
inverse_vocab = dict(enumerate(vocabulary))

decoded_sentence = " ".join(inverse_vocab[int(i)] for i in encoded_sentence)

print(decoded_sentence)

i write rewrite and [UNK] rewrite again


##### How Modern NLP Models Represent and Use Word Order

Words are easy to encode as categorical features, but encoding word order is the real challenge in NLP.

Word order varies across languages and even within sentences, making its relationship to meaning complex and non-linear.

Three major modeling approaches exist:

#1 - Ignore order → Bag-of-Words

#2 - Process sequentially → RNNs

#3 - Order-agnostic but position-aware → Transformers

RNNs and Transformers are sequence models because they incorporate word order, while early NLP relied mostly on bag-of-words.


### Sentiment analysis using IMDB dataset - with bag-of-words and RNN models

In [ ]:
# !curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
# !tar -xf aclImdb_v1.tar.gz

import os
import pathlib
import shutil
import random

### prepare a validation set by setting apart 20% of the training text files in a new directory, aclImdb/val

# Set up directory paths
base_dir = pathlib.Path("dataset/aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

# Create validation directories (neg, pos)
for category in ("neg", "pos"):
    os.makedirs(val_dir / category, exist_ok=True)

    # List and shuffle training files
    files = os.listdir(train_dir / category)
    random.Random(1337).shuffle(files)

    # Compute number of validation samples (20%)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]

    # Move files from train → val
    for fname in val_files:
        shutil.move(
            train_dir / category / fname,
            val_dir / category / fname
        )

In [20]:
from tensorflow import keras
batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
"dataset/aclImdb/train", batch_size=batch_size
)
val_ds = keras.utils.text_dataset_from_directory(
"dataset/aclImdb/val", batch_size=batch_size
)
test_ds = keras.utils.text_dataset_from_directory(
"dataset/aclImdb/test", batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [21]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

inputs.shape: (32,)
inputs.dtype: <dtype: 'string'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor(b'This is one of the most boring movies I have ever seen, its horrible. Christopher Lee is good but he is hardly in it, the only the good part is the opening scene.<br /><br />Don\'t be fooled by the title. "End of the World" is truly a bad movie, I stopped watching it close to the end it was so bad, only for die hard b-movie fans that have the brain to stand this vomit.', shape=(), dtype=string)
targets[0]: tf.Tensor(0, shape=(), dtype=int32)


##### Processing using bag of words

##### Unigram

In [ ]:
from tensorflow.keras.layers import TextVectorization

# TextVectorization is a powerful preprocessing layer in TensorFlow
# that transforms raw text into numerical representations that a model
# can understand. Here, we configure it for a simple bag-of-words style
# representation (multi-hot vector).

text_vectorization = TextVectorization(
    
    # ngrams=1 means we are only looking at single words ("unigrams").
    # If we set ngrams=2 or higher, the layer would also generate bigrams
    # or trigrams (word pairs or triples), capturing more context.
    ngrams=1,

    # max_tokens specifies the maximum vocabulary size.
    # The layer will keep only the 'top 20,000 most frequent words'
    # from the training data and ignore the rest.
    # This helps prevent the model from becoming too large and slow.
    max_tokens=20000,

    # output_mode="multi_hot" means each text will be converted into
    # a multi-hot encoded vector of size `max_tokens`.
    #
    # Example:
    #     Text: "good movie"
    # Multi-hot vector:
    #     [0, 1, 0, 1, 0, 0, 0, ...]
    #
    # A value becomes 1 if the corresponding word exists in the text,
    # and 0 if it does not.
    #
    # Unlike "count" mode (which counts occurrences), "multi_hot"
    # only cares about whether the word appeared at least once.
    output_mode="multi_hot",
)

# ---------------------------------------------------------
# 1. Extract ONLY the input text from the (x, y) pairs
# ---------------------------------------------------------
# train_ds contains pairs: (text, label)
# But when preparing the vocabulary using adapt(), we only need the raw text
# (because the labels do not matter for vocabulary building).
#
# The map() function transforms each element of the dataset.
# lambda x, y: x   → for each (x, y), we return only x (the text input).
#
# Example:
#   Input: ("This movie was great!", 1)
#   Output: "This movie was great!"
#
text_only_train_ds = train_ds.map(lambda x, y: x)


# ---------------------------------------------------------
# 2. Build vocabulary using ADAPT
# ---------------------------------------------------------
# text_vectorization.adapt() scans through ALL the text in the dataset.
# It learns:
#   - which words occur
#   - how often they occur
#   - creates a fixed-size vocabulary of the top max_tokens words
#
# adapt() MUST be done BEFORE applying the vectorizer to the datasets.
# After this step, the vectorizer knows how to convert words → token indices.
#
text_vectorization.adapt(text_only_train_ds)


# ---------------------------------------------------------
# 3. Vectorize TRAIN dataset
# ---------------------------------------------------------
# Now we convert each example (x, y) into:
#    (vectorized_x, y)
#
# text_vectorization(x) converts raw text into a multi-hot vector.
# num_parallel_calls=4 uses four parallel CPU threads to speed up processing.
#
# Example:
#    "good movie"  →  [0,1,0,1,0,0,...]
#
binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)


# ---------------------------------------------------------
# 4. Vectorize VALIDATION dataset
# ---------------------------------------------------------
# We apply the same vectorizer to validation data.
# IMPORTANT: We DO NOT call adapt() again, because the vocabulary must
# remain consistent with training.
#
binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)


# ---------------------------------------------------------
# 5. Vectorize TEST dataset
# ---------------------------------------------------------
# Same process for the test set.
# The vectorizer converts text into multi-hot vectors using the SAME vocabulary
# learned during training.
#
binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

2025-11-29 21:24:51.207886: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [28]:
for inputs, targets in binary_1gram_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

inputs.shape: (32, 20000)
inputs.dtype: <dtype: 'int64'>
targets.shape: (32,)
targets.dtype: <dtype: 'int32'>
inputs[0]: tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
targets[0]: tf.Tensor(1, shape=(), dtype=int32)


In [29]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=16):
    """
    Build a simple binary classification model using a 
    multi-hot encoded input vector.
    
    Args:
        max_tokens (int): Size of the input vector (vocabulary size).
        hidden_dim (int): Number of units in the hidden Dense layer.
    
    Returns:
        Compiled Keras model ready for training.
    """

    # Input shape = (max_tokens,), i.e., a multi-hot vector per sample
    inputs = keras.Input(shape=(max_tokens,))

    # Hidden dense layer to learn patterns from the multi-hot input
    x = layers.Dense(hidden_dim, activation="relu")(inputs)

    # Dropout to reduce overfitting
    x = layers.Dropout(0.5)(x)

    # Final output layer: sigmoid for binary classification
    outputs = layers.Dense(1, activation="sigmoid")(x)

    # Create model
    model = keras.Model(inputs, outputs)

    # Compile with binary crossentropy and accuracy metric
    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [30]:
# Build the model
model = get_model()

# View model architecture
model.summary()

# Save only the best model during training
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "binary_1gram.keras",
        save_best_only=True
    )
]

# Train the model
# .cache() improves performance by keeping dataset in memory
model.fit(
    binary_1gram_train_ds.cache(),
    validation_data=binary_1gram_val_ds.cache(),
    epochs=10,
    callbacks=callbacks
)

# Load the best saved model from disk
model = keras.models.load_model("binary_1gram.keras")

# Evaluate on the test dataset
test_acc = model.evaluate(binary_1gram_test_ds)[1]
print(f"Test acc: {test_acc:.3f}")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8301 - loss: 0.3994 - val_accuracy: 0.8984 - val_loss: 0.2639
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8990 - loss: 0.2701 - val_accuracy: 0.9008 - val_loss: 0.2600
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9164 - loss: 0.2368 - val_accuracy: 0.9012 - val_loss: 0.2708
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9220 - loss: 0.2280 - val_accuracy: 0.9008 - val_loss: 0.2823
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9259 - loss: 0.2231 - val_accuracy: 0.9008 - val_loss: 0.2956
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 970us/step - accuracy: 0.9303 - loss: 0.2098 - val_accuracy: 0.8976 - val_loss: 0.3089
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 984us/step - accuracy: 0.9315 - loss: 0.2071 - val_accuracy: 0.8952 - val_loss: 0.3214
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 947us/step - accuracy: 0.9366 - loss: 0.2038 - val_accura

##### Bigram

In [31]:
text_vectorization = TextVectorization(
ngrams=2,
max_tokens=20000,
output_mode="multi_hot",
)

In [32]:
text_vectorization.adapt(text_only_train_ds)
binary_2gram_train_ds = train_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
binary_2gram_val_ds = val_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
binary_2gram_test_ds = test_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
model = get_model()
model.summary()
callbacks = [
keras.callbacks.ModelCheckpoint("binary_2gram.keras",
save_best_only=True)
]

2025-11-29 21:35:07.944261: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
model.fit(binary_2gram_train_ds.cache(),
validation_data=binary_2gram_val_ds.cache(),
epochs=10,
callbacks=callbacks)
model = keras.models.load_model("binary_2gram.keras")
print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8375 - loss: 0.3899 - val_accuracy: 0.9016 - val_loss: 0.2565
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9108 - loss: 0.2489 - val_accuracy: 0.9088 - val_loss: 0.2456
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9299 - loss: 0.2048 - val_accuracy: 0.9054 - val_loss: 0.2602
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9398 - loss: 0.1948 - val_accuracy: 0.9040 - val_loss: 0.2701
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9463 - loss: 0.1812 - val_accuracy: 0.9038 - val_loss: 0.2881
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9485 - loss: 0.1767 - val_accuracy: 0.9034 - val_loss: 0.2930
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9534 - loss: 0.1656 - val_accuracy: 0.8990 - val_loss: 0.3114
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 957us/step - accuracy: 0.9535 - loss: 0.1687 - val_accuracy: 

##### Count Vectorizer

In [34]:
text_vectorization = TextVectorization(
ngrams=2,
max_tokens=20000,
output_mode="count"
)

In [35]:
text_vectorization.adapt(text_only_train_ds)
binary_2gram_train_ds = train_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
binary_2gram_val_ds = val_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
binary_2gram_test_ds = test_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
model = get_model()
model.summary()
callbacks = [
keras.callbacks.ModelCheckpoint("binary_2gram.keras",
save_best_only=True)
]

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.fit(binary_2gram_train_ds.cache(),
validation_data=binary_2gram_val_ds.cache(),
epochs=10,
callbacks=callbacks)
model = keras.models.load_model("binary_2gram.keras")
print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7707 - loss: 0.4857 - val_accuracy: 0.8972 - val_loss: 0.2951
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8597 - loss: 0.3510 - val_accuracy: 0.8984 - val_loss: 0.2727
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8842 - loss: 0.3055 - val_accuracy: 0.9038 - val_loss: 0.2660
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8947 - loss: 0.2796 - val_accuracy: 0.8914 - val_loss: 0.2717
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9021 - loss: 0.2604 - val_accuracy: 0.8960 - val_loss: 0.2846
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 999us/step - accuracy: 0.9061 - loss: 0.2548 - val_accuracy: 0.8872 - val_loss: 0.2793
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 999us/step - accuracy: 0.9114 - loss: 0.2407 - val_accuracy: 0.8818 - val_loss: 0.2916
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 964us/step - accuracy: 0.9172 - loss: 0.2309 - val_accura

##### TF-IDF Vectorizer

<pre>
Words that appear many times in a single document are usually important for understanding what that document is about.
But words that appear everywhere across all documents (like the, a, and) carry very little meaning — they don’t help            distinguish one document from another.
In contrast, rare words that appear in only a few documents (like names or technical terms) are highly 
informative and help identify the topic.

TF-IDF combines both ideas by giving each term a weight based on:
TF (Term Frequency): how often the term appears in the current document
IDF (Inverse Document Frequency): how rare the term is across the whole dataset

The final score is:
    TF-IDF = Term Frequency / Document Frequency,meaning important, rare words get high weight, 
    and common words get low weight.
</pre>

In [37]:
text_vectorization = TextVectorization(
ngrams=2,
max_tokens=20000,
output_mode="tf_idf",
)

In [38]:
text_vectorization.adapt(text_only_train_ds)
binary_2gram_train_ds = train_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
binary_2gram_val_ds = val_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
binary_2gram_test_ds = test_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
model = get_model()
model.summary()
callbacks = [
keras.callbacks.ModelCheckpoint("binary_2gram.keras",
save_best_only=True)
]

2025-11-29 21:42:39.124863: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
model.fit(binary_2gram_train_ds.cache(),
validation_data=binary_2gram_val_ds.cache(),
epochs=10,
callbacks=callbacks)
model = keras.models.load_model("binary_2gram.keras")
print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7811 - loss: 0.4684 - val_accuracy: 0.9026 - val_loss: 0.2692
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 924us/step - accuracy: 0.8670 - loss: 0.3208 - val_accuracy: 0.8876 - val_loss: 0.2693
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 949us/step - accuracy: 0.8796 - loss: 0.2918 - val_accuracy: 0.8906 - val_loss: 0.2695
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8896 - loss: 0.2725 - val_accuracy: 0.8994 - val_loss: 0.2718
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8993 - loss: 0.2581 - val_accuracy: 0.8916 - val_loss: 0.2827
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9028 - loss: 0.2475 - val_accuracy: 0.8792 - val_loss: 0.2903
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 971us/step - accuracy: 0.9013 - loss: 0.2430 - val_accuracy: 0.8742 - val_loss: 0.3155
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 941us/step - accuracy: 0.9038 - loss: 0.2451 - val_accu

##### Exporting a model that processes raw strings

In [40]:
# Create an input placeholder that expects raw strings.
# shape=(1,) means each input is a single string (a single text sample),
# not a sequence of words. dtype="string" tells Keras that the input
# will be text, not numbers.
inputs = keras.Input(shape=(1,), dtype="string")

# Pass the raw string input through the TextVectorization layer.
# This layer:
#   - standardizes the text (lowercasing, punctuation removal, etc.)
#   - splits it into tokens
#   - converts it into a multi-hot vector (because output_mode="multi_hot")
# This turns human-readable text → numeric vector for the model.
processed_inputs = text_vectorization(inputs)

# Feed the vectorized text into the trained model.
# The earlier model expects multi-hot vectors, so we feed the output
# of text_vectorization directly into it.
# This produces the final prediction (e.g., sentiment score).
outputs = model(processed_inputs)

# Build an inference model that goes end-to-end:
#     Raw text → TextVectorization → Dense layers → Prediction
#
# This combined model allows you to call inference_model.predict()
# using raw text without manually vectorizing it first.
inference_model = keras.Model(inputs, outputs)

In [63]:
import tensorflow as tf

# Convert raw text into a TensorFlow tensor.
# Each element must be wrapped in an inner list because the model
# expects inputs with shape (batch_size, 1), where each sample is a string.
raw_text_data = tf.convert_to_tensor([
    ["This movie is a worst one. Nobody should watch it"],
])

# Pass the raw text directly to the inference model.
# The inference model handles:
#   1. Text preprocessing (via TextVectorization)
#   2. Converting text into a multi-hot vector
#   3. Feeding it through the trained model
#   4. Outputting a probability score
predictions = inference_model(raw_text_data)

# Print the raw model output for the first (and only) sample.
# This is a value between 0 and 1:
#   - Close to 1 → highly positive sentiment
#   - Close to 0 → highly negative sentiment
print(predictions[0])

# Convert the probability (0–1) into a percentage and print.
print(f"{float(predictions[0] * 100):.2f} percent positive")

tf.Tensor([0.4316379], shape=(1,), dtype=float32)
43.16 percent positive


### Sequence model approach

No feature engineering

Let model figure out the features

In [ ]:
from tensorflow.keras import layers

max_length = 600
max_tokens = 20000

### Preparing the data

text_vectorization = TextVectorization(
    max_tokens=20000,
    output_mode='int',
    output_sequence_length=max_length
)

In [44]:
text_vectorization.adapt(text_only_train_ds)

In [48]:
len(text_vectorization.get_vocabulary())

20000

In [49]:
int_train_ds = train_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
int_val_ds = val_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)
int_test_ds = test_ds.map(
lambda x, y: (text_vectorization(x), y),
num_parallel_calls=4)

In [52]:
import keras
from keras import layers, ops

max_tokens = 20000

inputs = keras.Input(shape=(None,), dtype="int64")

# Use Keras ops instead of tf.one_hot
vector = ops.one_hot(inputs, num_classes=max_tokens)

x = layers.Bidirectional(layers.LSTM(32))(vector)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ one_hot (OneHot)                │ (None, None, 20000)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │     5,128,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,128,513 (19.56 MB)

 Trainable params: 5,128,513 (19.56 MB)

 Non-trainable params: 0 (0.00 B)

In [53]:
callbacks = [
keras.callbacks.ModelCheckpoint("one_hot_bidir_lstm.keras",
save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=10,
callbacks=callbacks)
model = keras.models.load_model("one_hot_bidir_lstm.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

Epoch 1/10
348/625 ━━━━━━━━━━━━━━━━━━━━ 4:41 1s/step - accuracy: 0.5542 - loss: 0.6770

KeyboardInterrupt: 

A first observation: this model trains very slowly, especially compared to the light-
weight model of the previous section. 

This is because our inputs are quite large: each
input sample is encoded as a matrix of size (600, 20000) (600 words per sample,
20,000 possible words). 

That’s 12,000,000 floats for a single movie review. Our bidirec-
tional LSTM has a lot of work to do. 

Second, the model only gets to 87% test accu-
racy—it doesn’t perform nearly as well as our (very fast) binary unigram model.

### Word embedding

One-hot vectors are all orthogonal to each other.

The geometric relationship between two word vectors
should reflect the semantic relationship between these words.

Word embeddings are vector representations of words that achieve exactly this.

The vectors obtained through one-hot encoding are binary, sparse (mostly
made of zeros), and very high-dimensional (the same dimensionality as the number of
words in the vocabulary).

Word embeddings are low-dimensional floating-point vectors
(that is, dense vectors, as opposed to sparse vectors).

<img src="./images/dl_rnn_11.png" />

Besides being dense representations, word embeddings are also structured representa-
tions, and their structure is learned from data. Similar words get embedded in close
locations.

In real-world word-embedding spaces, common examples of meaningful geometric
transformations are “gender” vectors and “plural” vectors. For instance, by adding a
“female” vector to the vector “king,” we obtain the vector “queen.” By adding a “plu-
ral” vector, we obtain “kings.” Word-embedding spaces typically feature thousands of
such interpretable and potentially useful vectors.

There are 2 ways to obtain word embeddings:

Learn word embeddings jointly with the main task you care about (such as doc-
ument classification or sentiment prediction). In this setup, you start with ran-
dom word vectors and then learn word vectors in the same way you learn the
weights of a neural network.

Load into your model word embeddings that were precomputed using a differ-
ent machine learning task than the one you’re trying to solve. These are called
pretrained word embedding

The perfect word-embedding space for an English-language movie-review
sentiment-analysis model may look different from the perfect embedding space for an
English-language legal-document classification model, because the importance of cer-
tain semantic relationships varies from task to task.

In [54]:
embedding_layer = layers.Embedding(input_dim=max_tokens, output_dim=256)

The Embedding layer is best understood as a dictionary that maps integer indices
(which stand for specific words) to dense vectors.

Word index -> Embedding layer -> Corresponding word vector

In [55]:
# ---------------------------------------------------------------
# 1. Define the input layer for the model
# ---------------------------------------------------------------
# The input consists of integer-encoded sequences.
# Each sequence is a list of token IDs produced by TextVectorization
# when output_mode="int".
#
# shape=(None,) means:
#   - variable-length sequence (different sentences can have different lengths)
#   - each element in the sequence is an int token ID
#
# dtype="int64" is required because token IDs are integers.
inputs = keras.Input(shape=(None,), dtype="int64")



# ---------------------------------------------------------------
# 2. Embedding Layer
# ---------------------------------------------------------------
# The Embedding layer converts each integer token ID into a dense vector
# of fixed size (here, 256 dimensions).
#
# Why Embeddings?
# - Instead of representing each word as a huge one-hot vector of length max_tokens,
#   Embedding learns a compact, meaningful representation.
# - Words with similar meaning end up with similar embedding vectors.
# - This helps the model learn semantic relationships between words.
#
# input_dim = max_tokens → size of vocabulary.
# output_dim = 256 → embedding vector size for each word.
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)



# ---------------------------------------------------------------
# 3. Bidirectional LSTM Layer
# ---------------------------------------------------------------
# LSTM reads sequences step-by-step and learns long-term dependencies.
# Bidirectional() wraps LSTM so that:
#   - one LSTM processes the sequence forward (left → right)
#   - one processes it backward (right → left)
#
# Why Bidirectional?
# - It allows the model to understand BOTH past and future context.
#   Example:
#      "not good" → meaning depends on the word before "good"
#      "good movie" → meaning depends on the word after "good"
#
# The output is a single vector summarizing the entire sequence.
x = layers.Bidirectional(layers.LSTM(32))(embedded)



# ---------------------------------------------------------------
# 4. Dropout Layer
# ---------------------------------------------------------------
# Dropout randomly turns off 50% of the neurons during training.
# This helps prevent overfitting by forcing the model to generalize
# rather than memorize the training data.
x = layers.Dropout(0.5)(x)



# ---------------------------------------------------------------
# 5. Output Layer
# ---------------------------------------------------------------
# Dense(1, activation="sigmoid") is used for binary classification.
#
# The sigmoid activation outputs a probability between 0 and 1:
#    - close to 1 → positive sentiment (or class = 1)
#    - close to 0 → negative sentiment (or class = 0)
outputs = layers.Dense(1, activation="sigmoid")(x)



# ---------------------------------------------------------------
# 6. Build and compile the model
# ---------------------------------------------------------------
# keras.Model connects inputs to outputs into a complete model.
model = keras.Model(inputs, outputs)

# Compile:
#   optimizer="rmsprop" – works well for RNNs
#   loss="binary_crossentropy" – correct for binary classification
#   metrics=["accuracy"] – track % of correct predictions
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display the model architecture
model.summary()



# ---------------------------------------------------------------
# 7. Set up a checkpoint callback
# ---------------------------------------------------------------
# ModelCheckpoint saves the best model during training.
# save_best_only=True ensures we don’t overwrite with worse versions.
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "embeddings_bidir_gru.keras",
        save_best_only=True
    )
]



# ---------------------------------------------------------------
# 8. Train the model
# ---------------------------------------------------------------
# int_train_ds and int_val_ds are integer-text datasets created earlier.
# epochs=10 → the model will see the full dataset 10 times.
model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10,
    callbacks=callbacks
)



# ---------------------------------------------------------------
# 9. Load the best model saved during training
# ---------------------------------------------------------------
model = keras.models.load_model("embeddings_bidir_gru.keras")



# ---------------------------------------------------------------
# 10. Evaluate the model on the test dataset
# ---------------------------------------------------------------
# model.evaluate returns [loss, accuracy]
test_acc = model.evaluate(int_test_ds)[1]
print(f"Test acc: {test_acc:.3f}")

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 73ms/step - accuracy: 0.7112 - loss: 0.5574 - val_accuracy: 0.8198 - val_loss: 0.4760
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 75ms/step - accuracy: 0.8439 - loss: 0.3968 - val_accuracy: 0.8570 - val_loss: 0.3415
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 74ms/step - accuracy: 0.8832 - loss: 0.3182 - val_accuracy: 0.8792 - val_loss: 0.3114
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 75ms/step - accuracy: 0.8985 - loss: 0.2761 - val_accuracy: 0.8896 - val_loss: 0.3098
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.9174 - loss: 0.2372 - val_accuracy: 0.8954 - val_loss: 0.2962
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.9325 - loss: 0.1992 - val_accuracy: 0.8540 - val_loss: 0.3681
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.9434 - loss: 0.1722 - val_accuracy: 0.8844 - val_loss: 0.3353
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.9545 - loss: 0.1413 - 

<pre>	

When training data is limited, you can’t learn good embeddings from scratch.
So you reuse pretrained word embeddings that already capture rich language structure.

This is similar to using pretrained CNNs in vision: 
you borrow generic, powerful features learned from massive datasets.

Pretrained embeddings (like Word2Vec and GloVe) are built from 
large-scale word co-occurrence statistics, often using unsupervised learning.

Word2Vec (Google, 2013) and GloVe (Stanford, 2014) became widely adopted 
because they capture meaningful semantic relationships (e.g., gender, analogies).

You can download these embeddings and load them 
directly into a Keras Embedding layer to build models that perform well even with small training datasets.

</pre>

##### Using GLOVE embeddings

#!wget http:/ /nlp.stanford.edu/data/glove.6B.zip

#!unzip -q glove.6B.zip

In [57]:
import numpy as np

# Path to the downloaded GloVe embedding file.
# Example file: "glove.6B.100d.txt"
# Each line in this file contains:
#   <word> <100 floating-point numbers>
# which together represent the embedding vector for that word.
path_to_glove_file = "dataset/glove.6B.100d.txt"

# Dictionary to store the mapping:
#   word → embedding vector (NumPy array)
embeddings_index = {}

# Open the GloVe file and read it line by line.
# The file is large, so streaming line-by-line is efficient.
with open(path_to_glove_file, encoding="utf-8") as f:
    for line in f:
        # Split each line into:
        #   - the first token: the word
        #   - the rest of the line: the vector values as a long string
        word, coefs = line.split(maxsplit=1)

        # Convert the vector string into a NumPy array of floats.
        # Example: "0.123 0.532 -0.019 ..." → array([0.123, 0.532, -0.019, ...])
        coefs = np.fromstring(coefs, dtype="f", sep=" ")

        # Store in dictionary
        embeddings_index[word] = coefs

# Print total number of words for which embeddings were loaded.
print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [58]:
# Dimension of the GloVe embeddings we loaded (100d in this example)
embedding_dim = 100

# ---------------------------------------------------------------
# 1. Retrieve the vocabulary from the TextVectorization layer
# ---------------------------------------------------------------
# text_vectorization.get_vocabulary() returns a list where:
#   index 0 → ""      (padding token)
#   index 1 → "[UNK]" (unknown token)
#   index 2 → "the"
#   index 3 → "movie"
#   ...
#
# These indices correspond to the integer token IDs used during training.
vocabulary = text_vectorization.get_vocabulary()

# Create a dictionary mapping:
#    word → integer index
#
# Example:
#    {"the": 2, "movie": 3, ...}
word_index = dict(zip(vocabulary, range(len(vocabulary))))



# ---------------------------------------------------------------
# 2. Build the embedding matrix
# ---------------------------------------------------------------
# embedding_matrix will have shape:
#    (max_tokens, embedding_dim)
#
# Each row i will contain the GloVe vector for the word whose index is i.
#
# If a word from our vocabulary is NOT found in GloVe,
# we keep its row as zeros (unknown embedding).
embedding_matrix = np.zeros((max_tokens, embedding_dim))

# Fill the embedding matrix
for word, i in word_index.items():

    # Only fill rows for words within our vocabulary limit
    if i < max_tokens:

        # Retrieve the pretrained GloVe vector for the word
        embedding_vector = embeddings_index.get(word)

        # If the word exists in GloVe, insert its vector into the matrix
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

# embedding_matrix is now ready to be loaded into a Keras Embedding layer!

In [59]:
# ---------------------------------------------------------------
# Create a Keras Embedding layer initialized with pretrained GloVe vectors
# ---------------------------------------------------------------
embedding_layer = layers.Embedding(
    
    # max_tokens defines the size of the vocabulary.
    # The embedding layer will have one row per token ID.
    input_dim=max_tokens,

    # embedding_dim is the number of features per word vector.
    # For GloVe.6B.100d → embedding_dim = 100.
    output_dim=embedding_dim,

    # Initialize the embedding weights using our precomputed embedding_matrix.
    # keras.initializers.Constant tells Keras:
    #     "Don't randomly initialize the embeddings—use THIS matrix instead."
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),

    # trainable=False freezes the embedding weights.
    # This means:
    #   - The model WILL NOT modify the GloVe vectors during training.
    #   - Useful when training data is small and we want to keep strong
    #     pretrained semantic information intact.
    #
    # If set to True, embeddings become fine-tunable.
    trainable=False,

    # mask_zero=True tells the embedding layer to treat token ID '0'
    # as a padding token and ignore it during LSTM processing.
    # This is essential when sequences are padded to equal length.
    mask_zero=True,
)

In [60]:
# ---------------------------------------------------------------
# 1. Define the Input Layer
# ---------------------------------------------------------------
# The model expects integer-encoded sequences as input.
# Each sequence is a list of word indices produced by TextVectorization.
#
# shape=(None,) means variable-length sequences — sentences may differ in length.
# dtype="int64" is required because the sequences contain integer token IDs.
inputs = keras.Input(shape=(None,), dtype="int64")



# ---------------------------------------------------------------
# 2. Pass Input Through the Pretrained GloVe Embedding Layer
# ---------------------------------------------------------------
# embedding_layer was created earlier using the GloVe embedding matrix.
#
# It converts each token ID into a 100-dimensional pretrained vector.
# Example:
#    [12, 87, 5] → [[0.21, -0.18, ...], [...], [...]]
#
# mask_zero=True ensures padded zeros are ignored by the LSTM.
embedded = embedding_layer(inputs)



# ---------------------------------------------------------------
# 3. Apply a Bidirectional LSTM
# ---------------------------------------------------------------
# LSTM processes sequences one timestep at a time.
# Bidirectional LSTM reads the sequence:
#   - Forward (left → right)
#   - Backward (right → left)
#
# This helps the model understand both previous and future context.
# Output is a single vector summarizing the whole sentence.
x = layers.Bidirectional(layers.LSTM(32))(embedded)



# ---------------------------------------------------------------
# 4. Add Dropout for Regularization
# ---------------------------------------------------------------
# Dropout randomly turns off 50% of neurons during training.
# This prevents overfitting and improves generalization,
# especially useful when using pretrained embeddings.
x = layers.Dropout(0.5)(x)



# ---------------------------------------------------------------
# 5. Add Final Output Layer for Binary Classification
# ---------------------------------------------------------------
# Dense(1, activation="sigmoid") outputs a value between 0 and 1.
# This represents the probability of positive sentiment.
outputs = layers.Dense(1, activation="sigmoid")(x)



# ---------------------------------------------------------------
# 6. Build and Compile the Model
# ---------------------------------------------------------------
model = keras.Model(inputs, outputs)

# Compile the model:
#   - rmsprop works well for RNN/LSTM tasks
#   - binary_crossentropy is required for binary classification
#   - accuracy tracks how often predictions are correct
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Print model architecture summary
model.summary()



# ---------------------------------------------------------------
# 7. Set Up Model Checkpoint Callback
# ---------------------------------------------------------------
# ModelCheckpoint saves the best-performing model during training.
# This ensures we keep the version with lowest validation loss.
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "glove_embeddings_sequence_model.keras",
        save_best_only=True
    )
]



# ---------------------------------------------------------------
# 8. Train the Model
# ---------------------------------------------------------------
# int_train_ds and int_val_ds are datasets where each sample is:
#    (integer_sequence, label)
#
# epochs=10 → the model trains over the entire dataset 10 times.
model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10,
    callbacks=callbacks
)



# ---------------------------------------------------------------
# 9. Load the Best Model Saved During Training
# ---------------------------------------------------------------
model = keras.models.load_model("glove_embeddings_sequence_model.keras")



# ---------------------------------------------------------------
# 10. Evaluate Model on Test Dataset
# ---------------------------------------------------------------
# Returns: [loss, accuracy]
test_acc = model.evaluate(int_test_ds)[1]
print(f"Test acc: {test_acc:.3f}")

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 100) │  2,000,000 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_9[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 64)        │     34,048 │ embedding_2[0][0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         65 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,034,113 (7.76 MB)

 Trainable params: 34,113 (133.25 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 99ms/step - accuracy: 0.6994 - loss: 0.5717 - val_accuracy: 0.7754 - val_loss: 0.4890
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 64s 102ms/step - accuracy: 0.7896 - loss: 0.4594 - val_accuracy: 0.8438 - val_loss: 0.3722
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 1118s 2s/step - accuracy: 0.8185 - loss: 0.4053 - val_accuracy: 0.8512 - val_loss: 0.3473
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 658s 1s/step - accuracy: 0.8415 - loss: 0.3698 - val_accuracy: 0.8568 - val_loss: 0.3367
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 148s 237ms/step - accuracy: 0.8539 - loss: 0.3426 - val_accuracy: 0.8718 - val_loss: 0.3131
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 64s 102ms/step - accuracy: 0.8676 - loss: 0.3196 - val_accuracy: 0.8776 - val_loss: 0.2998
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 64s 103ms/step - accuracy: 0.8740 - loss: 0.3024 - val_accuracy: 0.8772 - val_loss: 0.2974
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 101ms/step - accuracy: 0.8802 - loss: 0.28